**This script contains the exploratory analysis including sensitivity and correlation analyses**

In [ ]:
#Importing libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import xicorpy
import numpy as np
import matplotlib as mpl


**User inputs**

In [ ]:
#Generic turbine for the sensitivity analysis
my_path = 'add_path_here'
GT_file = 'add_GT_file_name_here'

#define the variable for sensitivity analysis
env_var = 'inFlowAngle' 

**Data prep for analysis**

In [ ]:
#----------------------------------------------- DATA PREPROCESSING  for OAT sensitivity analysis--------------------------------------------

#Consolodating the dataset into Python (and describing)
#reading in the dataset for generic turbine 

GT_df = pd.read_csv(GT_file)

#extracting the GT name from the file name
GT_name = GT_file.split('_')[0]

#grouping by env variables
GT_df = (
    GT_df
    .groupby([
        'windSpeed',
        'iRef',
        'shearExp',
        'density',
        'inFlowAngle',
        'node',
        'load',
        'wohler'
    ])['damage']
    .mean()
    .reset_index()
)

#exluding these nodes because they are not essential for analysis
excluded_nodes = {'stationary_hub', 'rotating_hub', 'blade_maxchord', 'tower_top'}

#Loop to extract different node/load/wohler subsets from the GT data compinations
GT_subsets = {}
for node in GT_df['node'].unique():
    for load in GT_df['load'].unique():
            for wohler in sorted(GT_df['wohler'].unique()):
                subset_name = f'{node}_{load}_{wohler}'
                if node not in excluded_nodes:
                    subset = GT_df[
                    (GT_df['node'] == node) &
                    (GT_df['load'] == load) &
                    (GT_df['wohler'] == wohler)
                    ]
                    # only store non-empty subsets
                    if not subset.empty:
                        GT_subsets[subset_name] = subset   
print(GT_subsets.keys())   #printing subsets

#creating arrays for each node with the different node/direction combinations
blade_root = []
blade_maxchord = []
tower_base = []
tower_top = []
stationary_hub = []
rotating_hub = []

for key in GT_subsets.keys():
     parts = key.split('_')
     node = f'{parts[0]}_{parts[1]}'
     if node == 'blade_root':
        blade_root.append(key)
     elif node == 'blade_maxchord':
        blade_maxchord.append(key)
     elif node == 'tower_base':
        tower_base.append(key)
     elif node == 'tower_top':
        tower_top.append(key)
     elif node == 'stationary_hub':
        stationary_hub.append(key)
     elif node == 'rotating_hub':
        rotating_hub.append(key)
        
#The environemntal parameters used
env_inputs = [
    'windSpeed',
    'iRef',
    'shearExp',
    'density',
    'inFlowAngle'
]


**Sensitivity Analysis**

A one at a time sensitivity analysis for each variable and DEL, will also colour the plots by other env variables to identify interaction effects. The variable for analysis must be defined first.

In [ ]:
#--------------------------------------------------------OAT SENSITIVITY-------------------------------------------------------------
#plotting settings for sensitivity

mpl.rcParams['font.size'] = 20
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['xtick.labelsize'] = 15
mpl.rcParams['ytick.labelsize'] = 15
mpl.rcParams['legend.fontsize'] = 18
mpl.rcParams['figure.dpi'] = 900

plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

env_vars = env_inputs

#variables to hold constant during sensitivity
group_vars = [
    var for var in env_vars
    if var != env_var 
]

#variables to colour by (all except the sensitivity variable)
colour_vars = [var for var in env_vars if var != env_var]

#colour maps for each variable
cmap_list = {
    'windSpeed': cm.plasma,
    'iRef': cm.viridis,
    'shearExp': cm.inferno,
    'density': cm.cividis,
    'inFlowAngle': cm.magma
}

label_map = {
    'windSpeed': 'Wind speed (m/s)',
    'iRef': 'Turbulence intensity',
    'shearExp': 'Shear exponent',
    'density': 'Density (kg/m³)',
    'inFlowAngle': 'Inflow angle (°)'
}

#only plotting some sample subsets
subsets_to_plot = ['blade_root_Mx_14', 'blade_root_My_14', 'tower_base_My_9']


# plotting
for subset_name, subset in GT_subsets.items():
    if subset_name not in subsets_to_plot:
        continue
    subset = subset.copy()  
    fig, axes = plt.subplots(len(colour_vars), 1, figsize=(6.27, 3.5 * len(colour_vars)), sharex=True)
    
    for ax, colour_var in zip(axes, colour_vars):

        #colour normalization
        colour_values = sorted(subset[colour_var].unique())
        norm = mcolors.Normalize(
            vmin=min(colour_values),
            vmax=max(colour_values)
        )
        cmap = cmap_list[colour_var] #creating a colour map for the colouring variable

        #grouping by all remaining fixed variables
        groups = subset.groupby(group_vars)

        #plotting each environmental-condition curve
        for name, group in groups:
            # ensure env_var changes
            if group[env_var].nunique() > 1:
                group = group.sort_values(env_var)
                colouring = group[colour_var].iloc[0]
                ax.plot(
                    group[env_var],
                    group['damage'],
                    color=cmap(norm(colouring)),
                    alpha=0.4  #making the lines slightly transparent
                )

        # colourbar
        sm = cm.ScalarMappable(
            cmap=cmap,
            norm=norm
        )

        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax)
        cbar.set_label(label_map[colour_var], fontsize=20, labelpad=10)
        cbar.set_ticks(colour_values)
        cbar.ax.tick_params(labelsize=15)

        ax.set_yscale('log')
        ax.set_ylabel('DEL (Nm)', labelpad=10)
        ax.grid(True)

    axes[-1].set_xlabel(label_map[env_var])
    fig.suptitle(f'{subset_name} - sensitivity to {env_var}', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{my_path}OAT_{env_var}_allcolours_{subset_name}.png', dpi=600)
    plt.show
  

**Correlation Analysis**
Finds the pearson, spearman and chatterjee correlation between each environmental variable and DEL and plots corresponding heatmaps. Performs the analysis across all GTs.

In [ ]:
# ------------------------------------------- CORRELATION CALCULATION (PEARSON and SPEARMAN and Xi )----------------------------------------------------
#A function which averages across damage values for different turbulence seeds
def power_average_damage(damage_values, wohler):
    return (np.mean(damage_values ** wohler)) ** (1/wohler)

#all GTs
GTs = [3,4,5,10,11,12]

all_pearson = []
all_spearman = []
all_chatterjee = []

#each GT file should have its own file in the path location
for GT in GTs:
    GT_name = f'GT{GT}'
    GT_path = f'{my_path}{GT_name}/'
    GT_df = pd.read_csv(f'{GT_path}{GT_name}_DB_list.csv')
    GT_df = GT_df.groupby(env_inputs + ['node', 'load', 'wohler']).apply(lambda x: pd.Series({'damage': power_average_damage(x['damage'].values, x.name[-1])}),include_groups=False).reset_index()

    GT_subsets = {}
    for node in GT_df['node'].unique():
        for load in GT_df['load'].unique():
            for wohler in sorted(GT_df['wohler'].unique()):
                subset_name = f'{node}_{load}_{wohler}'
                if node not in excluded_nodes:
                    subset = GT_df[
                        (GT_df['node'] == node) &
                        (GT_df['load'] == load) &
                        (GT_df['wohler'] == wohler)
                    ]
                    if not subset.empty:
                        GT_subsets[subset_name] = subset

    correlation_results_pearson = []
    correlation_results_spearman = []
    correlation_results_chatterjee = []

    # calculating the correlation coefficients for each env variable for each GT
    for name, subset in GT_subsets.items():
        for input_var in env_inputs:
            corr, pval = pearsonr(subset[input_var], subset['damage'])
            corr2, pval2 = spearmanr(subset[input_var], subset['damage'])
            xi, pval3 = xicorpy.compute_xi_correlation(subset[input_var], subset['damage'], get_p_values=True)
            xi = float(xi.iloc[0, 0])
            pval3 = float(pval3.iloc[0, 0])

            correlation_results_pearson.append({
                'subset': name,
                'variable': input_var,
                'pearson_r': corr,
                'p_value': pval
            })
            correlation_results_spearman.append({
                'subset': name,
                'variable': input_var,
                'spearman_r': corr2,
                'p_value': pval2
            })
            correlation_results_chatterjee.append({
                'subset': name,
                'variable': input_var,
                'chatterjee_r': xi,
                'p_value': pval3
            })

    # convert to pivot for this GT
    pearson_pivot = pd.DataFrame(correlation_results_pearson).pivot(index='variable', columns='subset', values='pearson_r')
    spearman_pivot = pd.DataFrame(correlation_results_spearman).pivot(index='variable', columns='subset', values='spearman_r')
    chatterjee_pivot = pd.DataFrame(correlation_results_chatterjee).pivot(index='variable', columns='subset', values='chatterjee_r')

    all_pearson.append(pearson_pivot)
    all_spearman.append(spearman_pivot)
    all_chatterjee.append(chatterjee_pivot)

# average across all GTs
pearson_df = sum(all_pearson) / len(all_pearson)
spearman_df = sum(all_spearman) / len(all_spearman)
chatterjee_df = sum(all_chatterjee) / len(all_chatterjee)


# print(pearson_df)
# print(spearman_df)
# print(chatterjee_df)

# heat map to visualise the most influential environmental inputs
env_order = ['inFlowAngle', 'shearExp', 'density', 'iRef', 'windSpeed']
col_order = list(pearson_df.columns)

def format_label(col):
    parts = col.split('_')
    node = 'blade\nroot' if parts[0] == 'blade' else 'tower\nbase'
    load = parts[2]
    wohler = parts[3]
    return f'{node}\n{load} {wohler}'

col_label_map = {col: format_label(col) for col in pearson_df.columns}
pearson_df = pearson_df.rename(columns=col_label_map)
spearman_df = spearman_df.rename(columns=col_label_map)
chatterjee_df = chatterjee_df.rename(columns=col_label_map)
col_order = list(pearson_df.columns)

#reformatting the labels
env_label_map = {
    'windSpeed': 'Wind speed',
    'iRef': 'Turbulence intensity',
    'shearExp': 'Shear exp',
    'density': 'Density',
    'inFlowAngle': 'Inflow angle'
}
pearson_df = pearson_df.rename(index=env_label_map)
spearman_df = spearman_df.rename(index=env_label_map)
chatterjee_df = chatterjee_df.rename(index=env_label_map)
env_order = ['Inflow angle', 'Shear exp', 'Density', 'Turbulence intensity', 'Wind speed']

#ensuring the plot font is correct
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

# Plotting the heatmaps of correlation averaged across GTs
# spearman
fig, ax = plt.subplots(figsize=(6.27, 3))
sns.heatmap(
    spearman_df.abs().loc[env_order, col_order],  # absolute for colour
    cmap='vlag', vmin=0, vmax=1,
    annot=spearman_df.loc[env_order, col_order].values,  # original for annotations
    fmt='.2f',
    annot_kws={'size': 7},
    ax=ax
)
plt.tight_layout()
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=7)
plt.yticks(rotation=0, fontsize=8)
plt.xticks(rotation=0, fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('')
plt.savefig(f'{my_path}spearman_overall_correlation_heatmap.pdf', dpi=600, pad_inches=0)
plt.show()
plt.close()

# pearson
fig, ax = plt.subplots(figsize=(6.27, 3))
sns.heatmap(
    pearson_df.abs().loc[env_order, col_order],  # absolute for colour
    cmap='vlag', vmin=0, vmax=1,
    annot=pearson_df.loc[env_order, col_order].values,  # original for annotations
    fmt='.2f',
    annot_kws={'size': 7},
    ax=ax
)
plt.tight_layout()
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=7)
plt.yticks(rotation=0, fontsize=8)
plt.xticks(rotation=0, fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('')
plt.savefig(f'{my_path}pearson_correlation_heatmap.pdf', dpi=600, pad_inches=0)
plt.show()
plt.close()

# chatterjee
fig, ax = plt.subplots(figsize=(6.27, 3))
sns.heatmap(chatterjee_df.loc[env_order, col_order], cmap='vlag', center=0.5, vmin=0, vmax=1, annot=True, annot_kws={'size': 7})
plt.tight_layout()
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=7)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=0, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.savefig(f'{my_path}chatterjee_correlation_heatmap.pdf', dpi=600, pad_inches=0)
plt.show()
plt.close()